# Two-Handed Signs Module — HamNoSys Pipeline

Detects and classifies **all two-handed relationship signs** from the HamNoSys *Two-handed* keyboard tab.

## Token Map

| Row | Tokens | Meaning |
|-----|--------|---------|
| ROW 1 | `hamparallel` `hamsequential` `hamseqleftright` `hamseqatloc` `hamseqoverlap` | Simultaneity / Sequencing |
| ROW 2 | `hamsymmlr` `hamsymmr` `hamsymmrl` `hamcircular` | Symmetry brackets |
| ROW 3 | `hamreplace` `hamsimultaneous` | Dominant / non-dominant role |
| ROW 4 | `hamholdsecond` | Passive hand hold |
| ROW 5 | `hamsecleftpar` `hamsecpar` `hamsecleftbrack` `hamsecbrack` `hamsecleftangle` `hamsecangle` `hamsecleftbrace` `hamsecbrace` `hamsecbar` | Secondary hand brackets |
| ROW 6 | `hamsecshape0` `hamsecshape1` `hamsecshape2` `hamsecshape3` `hamsecshape4` `hamsecshape5` `hamsecshape6` | Passive hand shape presets |
| ROW 7 | `hamsecloc_comma` `hamsecloc_dot` `hamsecloc_bar` `hamsecloc_q` `hamsecloc_cup` | Secondary location qualifiers |


In [ ]:
# Uncomment below if running on Colab:
# from google.colab import drive; drive.mount('/content/drive')
# !pip install mediapipe opencv-python numpy


In [ ]:
import os, sys, cv2, numpy as np
from collections import Counter, deque
import mediapipe as mp

_DIR = os.path.abspath('.')
if _DIR not in sys.path:
    sys.path.insert(0, _DIR)

try:
    from shared_landmarks import get_video_landmarks
    _USE_SHARED = True
    print('[OK] shared_landmarks imported')
except ImportError:
    _USE_SHARED = False
    print('[WARN] shared_landmarks not found - using direct MediaPipe')

_mp_hands = mp.solutions.hands


## Token Catalogues — All 36 HamNoSys Two-handed Tokens


In [ ]:
# ROW 1 - Simultaneity / Sequencing
TWO_HAND_SEQ_TOKENS = [
    'hamparallel',       # both hands active simultaneously
    'hamsequential',     # left hand then right hand
    'hamseqleftright',   # sequential left to right
    'hamseqatloc',       # at same location
    'hamseqoverlap',     # overlapping sequence
]

# ROW 2 - Symmetry / Relationship brackets
TWO_HAND_SYMM_TOKENS = [
    'hamsymmlr',   # symmetry left-right mirror (dominant drives)
    'hamsymmr',    # right-dominant symmetric motion
    'hamsymmrl',   # symmetry right-left mirror
    'hamcircular', # circular / alternating bilateral
]

# ROW 3 - Dominant / Non-dominant hand operators
TWO_HAND_ROLE_TOKENS = [
    'hamreplace',      # replaces previous hand configuration
    'hamsimultaneous', # both hands act simultaneously (explicit)
]

# ROW 4 - Secondary hand hold
TWO_HAND_HOLD_TOKENS = [
    'hamholdsecond',  # passive / non-dominant hand holds
]

# ROW 5 - Secondary hand articulator brackets
TWO_HAND_SEC_BRACKET_TOKENS = [
    'hamsecleftpar',    # ( open passive-hand description
    'hamsecpar',        # ) close passive-hand description
    'hamsecleftbrack',  # [ open secondary location
    'hamsecbrack',      # ] close secondary location
    'hamsecleftangle',  # < open secondary angle modifier
    'hamsecangle',      # > close secondary angle modifier
    'hamsecleftbrace',  # { open secondary shape group
    'hamsecbrace',      # } close secondary shape group
    'hamsecbar',        # | separator dominant/passive segments
]

# ROW 6 - Passive hand shape presets
TWO_HAND_SHAPE_TOKENS = [
    'hamsecshape0',  # square / palm-down flat shape
    'hamsecshape1',  # open / neutral shape
    'hamsecshape2',  # index finger only (1)
    'hamsecshape3',  # index + middle (2)
    'hamsecshape4',  # three fingers (3)
    'hamsecshape5',  # four fingers (4)
    'hamsecshape6',  # all fingers / flat (5)
]

# ROW 7 - Secondary body location qualifiers
TWO_HAND_LOC_TOKENS = [
    'hamsecloc_comma',  # minor location shift
    'hamsecloc_dot',    # at-same-point qualifier
    'hamsecloc_bar',    # perpendicular plane qualifier
    'hamsecloc_q',      # uncertain / context-dependent location
    'hamsecloc_cup',    # cupped / below location
]

ALL_TWO_HAND_TOKENS = (
    TWO_HAND_SEQ_TOKENS + TWO_HAND_SYMM_TOKENS +
    TWO_HAND_ROLE_TOKENS + TWO_HAND_HOLD_TOKENS +
    TWO_HAND_SEC_BRACKET_TOKENS + TWO_HAND_SHAPE_TOKENS +
    TWO_HAND_LOC_TOKENS
)
print(f'Total two-handed tokens: {len(ALL_TWO_HAND_TOKENS)}')
print(ALL_TWO_HAND_TOKENS)


## Geometric Helpers


In [ ]:
def _lm_to_arr(lm_obj):
    if lm_obj is None: return None
    try: pts = [(lm.x, lm.y, lm.z) for lm in lm_obj.landmark]
    except AttributeError: return None
    return np.array(pts, dtype=np.float32)

def _wrist(arr): return arr[0] if arr is not None else None

def _palm_center(arr):
    if arr is None: return None
    return np.mean(arr[[0,5,9,13,17]], axis=0)

def _dist(a, b):
    if a is None or b is None: return 9999.0
    return float(np.linalg.norm(a - b))

def _finger_states(arr):
    CHAINS = {'thumb':(1,2,3,4),'index':(5,6,7,8),
              'middle':(9,10,11,12),'ring':(13,14,15,16),'pinky':(17,18,19,20)}
    if arr is None: return {}
    states = {}
    for name,(mcp,pip,dip,tip) in CHAINS.items():
        d = _dist(arr[mcp],arr[tip])
        c = _dist(arr[mcp],arr[pip])+_dist(arr[pip],arr[dip])+_dist(arr[dip],arr[tip])
        r = d/(c+1e-8)
        states[name] = 'straight' if r>=0.76 else ('bent' if r>=0.50 else 'closed')
    return states

def _extended_finger_count(arr):
    if arr is None: return 0
    return sum(1 for k,v in _finger_states(arr).items() if k!='thumb' and v=='straight')

print('Helpers ready.')


## TwoHandAnalyzer — Per-Frame and Aggregate Classification


In [ ]:
class TwoHandAnalyzer:
    def __init__(self, history_len=500):
        self._history = deque(maxlen=history_len)
        self._seq_buf = deque(maxlen=10)

    def analyse_frame(self, left_arr, right_arr):
        result = {'symmetry':None,'sequencing':None,'role':None,
                  'hold':None,'sec_shape':None,'sec_loc':None}
        both  = (left_arr is not None) and (right_arr is not None)
        only_r= (right_arr is not None) and (left_arr is None)
        only_l= (left_arr is not None)  and (right_arr is None)

        # --- 1. Symmetry ---
        if both:
            lw=_wrist(left_arr); rw=_wrist(right_arr)
            xm=(lw[0]+rw[0])/2.0
            xe=abs((lw[0]-xm)+(rw[0]-xm))
            yd=abs(lw[1]-rw[1])
            fm=abs(_extended_finger_count(left_arr)-_extended_finger_count(right_arr))<=1
            if   xe<0.08 and yd<0.10 and fm: result['symmetry']='hamsymmlr'
            elif xe<0.15 and yd<0.15 and fm: result['symmetry']='hamsymmr'

        # --- 2. Sequencing ---
        if both:
            d=_dist(_palm_center(left_arr),_palm_center(right_arr))
            result['sequencing']='hamseqatloc' if d<0.05 else 'hamparallel'
        elif only_r or only_l:
            result['sequencing']='hamsequential'

        # --- 3. Passive Hand Shape Preset ---
        passive=left_arr if both else None
        if passive is not None:
            n=_extended_finger_count(passive)
            fs=_finger_states(passive)
            ac=all(v=='closed' for k,v in fs.items() if k!='thumb')
            shapes=['hamsecshape0','hamsecshape1','hamsecshape2',
                    'hamsecshape3','hamsecshape4','hamsecshape5','hamsecshape6']
            result['sec_shape']=shapes[0] if ac else (shapes[min(n,5)+1] if n>0 else shapes[1])

        # --- 4. Hold Detection ---
        if both and len(self._seq_buf)>=5:
            rec=list(self._seq_buf)[-5:]
            lws=[r[0] for r in rec if r[0] is not None]
            rws=[r[1] for r in rec if r[1] is not None]
            if len(lws)>=3 and len(rws)>=3:
                lm_=np.std(np.array(lws)[:,:2],axis=0).mean()
                rm_=np.std(np.array(rws)[:,:2],axis=0).mean()
                if lm_<0.008 and rm_>0.015: result['hold']='hamholdsecond'

        # --- 5. Secondary Location ---
        if both:
            lc=_palm_center(left_arr); rc=_palm_center(right_arr)
            dh=_dist(lc,rc); zd=abs(float(lc[2]-rc[2]))
            if   zd>0.08:  result['sec_loc']='hamsecloc_bar'
            elif dh<0.04:  result['sec_loc']='hamsecloc_dot'
            elif dh<0.12:  result['sec_loc']='hamsecloc_comma'

        self._history.append({'left_present':left_arr is not None,
                              'right_present':right_arr is not None,
                              'result':result.copy()})
        lw_p=_wrist(left_arr)[:3]  if left_arr  is not None else None
        rw_p=_wrist(right_arr)[:3] if right_arr is not None else None
        self._seq_buf.append((lw_p,rw_p))
        return result

    def aggregate(self):
        if not self._history: return _empty_result()
        sc=Counter(); qc=Counter(); hc=Counter(); shc=Counter(); lc=Counter()
        for f in self._history:
            r=f['result']
            if r['symmetry']:   sc[r['symmetry']]   +=1
            if r['sequencing']: qc[r['sequencing']]  +=1
            if r['hold']:       hc[r['hold']]          +=1
            if r['sec_shape']:  shc[r['sec_shape']]    +=1
            if r['sec_loc']:    lc[r['sec_loc']]        +=1
        n=max(len(self._history),1)
        sym=None
        if sc:
            ts,tn=sc.most_common(1)[0]
            if tn>0.40*n: sym=ts
        if sc.get('hamsymmlr',0)>0.2*n and sc.get('hamsymmr',0)>0.2*n: sym='hamcircular'
        seq=qc.most_common(1)[0][0]  if qc  else 'hamparallel'
        hld=hc.most_common(1)[0][0]  if hc  and hc.most_common(1)[0][1]>3 else None
        shp=shc.most_common(1)[0][0] if shc else None
        loc=lc.most_common(1)[0][0]  if lc  else None
        bf=sum(1 for f in self._history if f['left_present'] and f['right_present'])
        is2=bf>0.25*n
        return {'is_two_handed':is2,'symmetry_token':sym,'sequencing_token':seq,
                'hold_token':hld,'sec_shape_token':shp,'sec_loc_token':loc,
                'per_frame':[f['result'] for f in self._history],
                'final':_build_final(sym,seq,hld,shp,loc,is2)}

def _empty_result():
    return {'is_two_handed':False,'symmetry_token':None,'sequencing_token':'hamparallel',
            'hold_token':None,'sec_shape_token':None,'sec_loc_token':None,
            'per_frame':[],'final':None}

def _build_final(sym,seq,hold,shape,loc,is2):
    if not is2: return None
    parts=[]
    if sym:               parts.append(sym)
    elif seq and seq!='hamparallel': parts.append(seq)
    if hold:              parts.append(hold)
    if shape:             parts.extend(['hamsecleftpar',shape,'hamsecpar'])
    if loc:               parts.append(loc)
    return ' '.join(parts) if parts else 'hamparallel'

print('TwoHandAnalyzer ready.')


## run_two_hands_module — Main Entry Point


In [ ]:
def run_two_hands_module(video_path):
    """
    Main entry point - same pattern as all other module wrappers.

    Returns dict:
        is_two_handed    bool
        symmetry_token   str or None
        sequencing_token str
        hold_token       str or None
        sec_shape_token  str or None
        sec_loc_token    str or None
        per_frame        list of per-frame dicts
        final            str or None
    """
    analyzer = TwoHandAnalyzer()

    if _USE_SHARED:
        frames_data = get_video_landmarks(video_path)
        for fd in frames_data:
            left_arr  = _lm_to_arr(fd.get('left_hand'))
            right_arr = _lm_to_arr(fd.get('right_hand'))
            analyzer.analyse_frame(left_arr, right_arr)
        return analyzer.aggregate()

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f'[TwoHands] Cannot open: {video_path}')
        return _empty_result()

    with _mp_hands.Hands(static_image_mode=False, max_num_hands=2,
                         model_complexity=1,
                         min_detection_confidence=0.5,
                         min_tracking_confidence=0.5) as hd:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = hd.process(rgb)
            left_arr = right_arr = None
            if res.multi_hand_landmarks and res.multi_handedness:
                for lm, handed in zip(res.multi_hand_landmarks, res.multi_handedness):
                    label = handed.classification[0].label
                    arr = np.array([(lp.x,lp.y,lp.z) for lp in lm.landmark],dtype=np.float32)
                    if label=='Right' and right_arr is None: right_arr=arr
                    elif label=='Left' and left_arr is None: left_arr=arr
            analyzer.analyse_frame(left_arr, right_arr)
    cap.release()
    return analyzer.aggregate()

print('run_two_hands_module ready.')


## inject_two_hands — Pipeline Integration Hook

Patches the HamNoSys token list from `combine_hamnosys()` with two-handed tokens at grammatically correct positions.


In [ ]:
def inject_two_hands(hamnosys_sequence_list, two_hands_result):
    """
    Insert two-handed tokens into the combined HamNoSys token sequence.

    Grammar positions:
      Symmetry / sequencing  -> slot 0 (before base handshape)
      hamsec* brackets       -> after body-location token
      secondary loc token    -> before movement token
      hamholdsecond          -> appended at end
    """
    if not two_hands_result.get('is_two_handed'): return hamnosys_sequence_list
    seq   = list(hamnosys_sequence_list)
    symm  = two_hands_result.get('symmetry_token')
    seqt  = two_hands_result.get('sequencing_token')
    hold  = two_hands_result.get('hold_token')
    shape = two_hands_result.get('sec_shape_token')
    loc   = two_hands_result.get('sec_loc_token')

    # 1. Symmetry / sequencing at slot 0
    if symm and symm not in seq:
        seq.insert(0, symm)
    elif seqt and seqt not in seq and seqt != 'hamparallel':
        seq.insert(0, seqt)

    # 2. Passive hand shape after body-location token
    if shape:
        loc_kw=('hamhead','hamforehead','hamchin','hamlips','hamchest',
                'hamshoulders','hamneck','hamear','hampalm','hamhandback')
        ins=None
        for i,tok in enumerate(seq):
            if any(tok.startswith(kw) for kw in loc_kw): ins=i
        if ins is not None:
            for j,t in enumerate(['hamsecleftpar',shape,'hamsecpar']):
                seq.insert(ins+1+j, t)
        else:
            seq.extend(['hamsecleftpar',shape,'hamsecpar'])

    # 3. Secondary location qualifier before movement
    if loc and loc not in seq:
        mov_kw=('hammove','hamswing','hamrepeat','hamcircle','hamflex','hamnod')
        ib=len(seq)
        for i,tok in enumerate(seq):
            if any(tok.startswith(kw) for kw in mov_kw): ib=i; break
        seq.insert(ib, loc)

    # 4. hamholdsecond at end
    if hold and hold not in seq: seq.append(hold)
    return seq

print('inject_two_hands ready.')


## Run on Your Video

Set `VIDEO_PATH` to your sign-language video file.


In [ ]:
VIDEO_PATH = 'Prompt_1.mp4'  # <-- change this

result = run_two_hands_module(VIDEO_PATH)

print('=' * 50)
print('TWO-HANDED MODULE RESULTS')
print('=' * 50)
print(f"  Is two-handed       : {result['is_two_handed']}")
print(f"  Symmetry token      : {result['symmetry_token']}")
print(f"  Sequencing token    : {result['sequencing_token']}")
print(f"  Hold token          : {result['hold_token']}")
print(f"  Passive shape token : {result['sec_shape_token']}")
print(f"  Sec location token  : {result['sec_loc_token']}")
print()
print(f"  FINAL token string  : {result['final']}")


## How to Connect with integration_pipeline.py


In [ ]:
INTEGRATION_SNIPPET = '''
# ── Add in integration_pipeline.py ──────────────────────────────────────────

# 1. Import at top:
from two_hands import run_two_hands_module, inject_two_hands

# 2. In generate_hamnosys() after the 10 existing modules:
two_hands = run_two_hands_module(video_path)
print("  [11/11] Two-Handed - done")

# 3. After combine_hamnosys() returns hamnosys_code:
token_list = hamnosys_code.split()
token_list = inject_two_hands(token_list, two_hands)
hamnosys_code = ' '.join(token_list)

# 4. In the return dict:
return hamnosys_code, {
    ...,  # existing keys
    'two_hands': two_hands,
}
'''
print(INTEGRATION_SNIPPET)
